---
title: 'Lab 4: Własne estymatory scikit-learn'
subtitle: Biblioteki Python w analizie danych
author: Tomasz Rodak
toc-title: Spis treści
jupyter: python3
---


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_4.ipynb)

Na wykładzie 2 poznaliśmy architekturę scikit-learn: estymatory, transformatory i predyktory ze spójnym interfejsem `fit()` / `transform()` / `predict()`. Widzieliśmy gotowe przykłady (`RegresjaLiniowa`, `Standaryzator`) i dowiedzieliśmy się, jakie konwencje musi spełniać estymator, aby współpracował z potokami i kroswalidacją.

W tym arkuszu zbudujemy od podstaw dwa estymatory: transformator i klasyfikator. Zaczniemy od najprostszej możliwej struktury i będziemy ją stopniowo rozbudowywać, testując na każdym etapie zgodność z API scikit-learn.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

## 1. Anatomia estymatora — przypomnienie

Każdy estymator scikit-learn opiera się na kilku konwencjach:

- **Hiperparametry** to argumenty `__init__`. Muszą być przypisane do atrybutów o dokładnie takich samych nazwach (`self.param = param`). Nie należy w `__init__` wykonywać żadnych obliczeń ani modyfikować wartości argumentów.
- **Atrybuty wyuczone** kończą się podkreślnikiem (`self.coef_`, `self.mean_`). Powstają wyłącznie w metodzie `fit()`.
- Metoda `fit()` zwraca `self`.
- **Walidacja danych** korzysta z funkcji `check_X_y`, `check_array`, `check_is_fitted` z `sklearn.utils.validation`.
- **Klasy bazowe**: `BaseEstimator` daje `get_params()` / `set_params()`, a mixiny dodają domyślne metody — `TransformerMixin` dodaje `fit_transform()`, `ClassifierMixin` dodaje `score()` (accuracy).

Poniżej zastosujemy te zasady w praktyce.


## 2. Własny transformator: `MedianImputer`

Braki danych (`NaN`) to częsty problem w zbiorach tabelarycznych. Jedną z prostszych strategii jest zastąpienie brakujących wartości medianą danej cechy, obliczoną na zbiorze treningowym.

Naszym celem jest zaimplementowanie transformatora `MedianImputer`, który:

- w `fit()` oblicza medianę każdej cechy (ignorując `NaN`),
- w `transform()` wypełnia braki wyznaczonymi medianami.

Na końcu porównamy wyniki z `sklearn.preprocessing.SimpleImputer(strategy='median')`.


### 2.1 Dane z brakami

Poniższy kod tworzy niewielki zbiór z celowo wprowadzonymi brakami — posłuży do testowania implementacji.


In [ ]:
rng = np.random.default_rng(42)

X_full = rng.standard_normal((8, 3))

# Wprowadź braki w losowych miejscach
X_missing = X_full.copy()
mask = rng.choice([True, False], size=X_full.shape, p=[0.25, 0.75])
X_missing[mask] = np.nan

print("Dane z brakami:")
print(X_missing)
print("\nLiczba braków w każdej kolumnie:", np.isnan(X_missing).sum(axis=0))

### 2.2 Implementacja klasy

Uzupełnij poniższy szkielet. Kilka wskazówek:

- `np.nanmedian(X, axis=0)` oblicza medianę kolumn, ignorując `NaN`.
- Do wypełniania braków: utwórz kopię tablicy, znajdź pozycje `NaN` za pomocą `np.isnan()` i podmień je na odpowiednie mediany. Przydatna może być konstrukcja `np.where(np.isnan(X), self.median_, X)`, która dla każdego elementu wybiera medianę (jeśli `NaN`) lub oryginalną wartość.
- Pamiętaj o walidacji danych. Funkcja `check_array` przyjmuje argument `force_all_finite` — domyślna wartość `True` odrzuci tablicę z `NaN`. Użyj `ensure_all_finite='allow-nan'`, aby na to pozwolić.

In [ ]:
class MedianImputer(BaseEstimator, TransformerMixin):
    """Wypełnia brakujące wartości (NaN) medianą z danych treningowych."""

    def fit(self, X, y=None):
        X = check_array(X, ensure_all_finite='allow-nan')
        self.n_features_in_ = X.shape[1]
        # Oblicz medianę każdej cechy, ignorując NaN
        # ...
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X, force_all_finite='allow-nan')
        # Wypełnij NaN medianami z fit()
        # ...

### 2.3 Weryfikacja

#### 2.3.1 Test ręczny

Dopasuj `MedianImputer` do `X_missing` i przetransformuj te same dane. Sprawdź, że:

1. Wynikowa tablica nie zawiera `NaN`.
2. Wartości, które nie były `NaN`, nie uległy zmianie.


#### 2.3.2 Porównanie z `SimpleImputer`

Porównaj wyniki swojego transformatora z `SimpleImputer(strategy='median')` z scikit-learn. Czy tablice wynikowe są identyczne?

*Wskazówka:* `np.allclose()` porównuje tablice z tolerancją numeryczną.


In [ ]:
from sklearn.impute import SimpleImputer


#### 2.3.3 Dane treningowe ≠ dane testowe

Ważna właściwość transformatora: mediany obliczone w `fit()` powinny być stosowane do *nowych* danych w `transform()`. Sprawdź to:

1. Dopasuj `MedianImputer` do `X_missing`.
2. Utwórz nową tablicę `X_new` (np. 3 wiersze) z innymi brakami.
3. Przetransformuj `X_new`. Czy użyte mediany pochodzą z `X_missing`, a nie z `X_new`?

*Wskazówka:* Wypisz `imputer.median_` i porównaj z medianami obliczonymi bezpośrednio z `X_new`.


## 3. Własny predyktor: `KNNClassifier`

Algorytm *k* najbliższych sąsiadów (*k-nearest neighbors*, k-NN) to jeden z najprostszych algorytmów klasyfikacji: aby sklasyfikować nowy punkt, znajdujemy $k$ najbliższych punktów treningowych i przypisujemy klasę, która wśród nich dominuje.


### 3.1 Algorytm

**Faza uczenia** (`fit`): zapamiętaj dane treningowe $X_{\text{train}}$ (kształt $(N, d)$) i etykiety $y_{\text{train}}$ (kształt $(N,)$).

**Faza predykcji** (`predict`): dla każdego punktu testowego $x$ z macierzy $X_{\text{test}}$ (kształt $(M, d)$):

1. Oblicz odległość od $x$ do każdego punktu treningowego — wynikiem jest wektor o długości $N$.
2. Znajdź indeksy $k$ punktów o najmniejszej odległości.
3. Odczytaj etykiety tych $k$ sąsiadów.
4. Przypisz klasę najczęściej występującą wśród sąsiadów (głosowanie większościowe).

**Wynik:** wektor predykcji o kształcie $(M,)$.


### 3.2 Macierz odległości (broadcasting)

Zanim przejdziemy do pełnej klasy, zaimplementujmy kluczowy krok obliczeniowy: macierz kwadratów odległości euklidesowych między każdym punktem testowym a każdym punktem treningowym.

Przeanalizuj transformację kształtów:

```
X_test:   (M, d)  →  (M, 1, d)
X_train:  (N, d)  →  (1, N, d)
różnica:              (M, N, d)
kwadrat + suma po osi 2:  (M, N)
```

Wynikiem jest tablica, w której element $(i, j)$ to $\|x^{\text{test}}_i - x^{\text{train}}_j\|^2$.

Wygeneruj dane testowe za pomocą `make_classification` i zaimplementuj obliczenie macierzy odległości:


In [ ]:
from sklearn.datasets import make_classification

X_train, y_train = make_classification(
    n_samples=200, n_features=4, n_informative=3,
    n_classes=3, n_clusters_per_class=1, random_state=42
)
X_test, y_test = make_classification(
    n_samples=50, n_features=4, n_informative=3,
    n_classes=3, n_clusters_per_class=1, random_state=7
)

In [ ]:
# Oblicz macierz kwadratów odległości (M, N)
# ...

Sprawdź kształt wyniku — powinien być `(50, 200)`.


### 3.3 Znajdowanie *k* najbliższych sąsiadów

Mając macierz odległości, potrzebujemy dla każdego wiersza (punktu testowego) znaleźć indeksy $k$ kolumn o najmniejszej wartości.

Funkcja `np.argpartition(a, k, axis=1)` częściowo sortuje tablicę wzdłuż osi 1 tak, że $k$ najmniejszych elementów trafia na pozycje `0, 1, ..., k-1` (ale niekoniecznie w kolejności). Jest znacznie szybsza niż pełne sortowanie — złożoność $O(N)$ zamiast $O(N \log N)$.

1. Użyj `np.argpartition` na macierzy odległości, aby znaleźć indeksy $k = 5$ najbliższych sąsiadów dla każdego punktu testowego. Wynikiem powinien być wycinek o kształcie $(M, k)$.
2. Odczytaj etykiety sąsiadów: `y_train[indeksy_najblizszych]`. Jaki jest kształt wyniku?


### 3.4 Głosowanie większościowe

Mając tablicę etykiet sąsiadów o kształcie $(M, k)$, dla każdego punktu testowego wybierz klasę, która występuje najczęściej. 

Jednym ze sposobów jest użycie `np.apply_along_axis` z funkcją, która wywołuje `np.bincount` i `np.argmax`:

```python
def most_common(labels):
    counts = np.bincount(labels)
    return np.argmax(counts)
```

Zastosuj tę funkcję do tablicy etykiet sąsiadów, aby uzyskać wektor predykcji o kształcie $(M,)$.

*Uwaga:* `np.bincount` wymaga danych typu całkowitego. Jeśli etykiety są typu `float`, użyj `.astype(int)`.


### 3.5 Implementacja klasy `KNNClassifier`

Połącz kroki 3.2–3.4 w klasę zgodną z API scikit-learn.


In [ ]:
class KNNClassifier(BaseEstimator, ClassifierMixin):
    """Klasyfikator k najbliższych sąsiadów."""

    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        # 1. Walidacja danych (check_X_y)
        # 2. Zapamiętaj dane treningowe i etykiety
        #    jako atrybuty wyuczone (z podkreślnikiem)
        # 3. Zapamiętaj unikalne klasy (self.classes_)
        # 4. Zapamiętaj liczbę cech (self.n_features_in_)
        # ...
        return self

    def predict(self, X):
        # 1. Sprawdź, czy model jest dopasowany (check_is_fitted)
        # 2. Walidacja danych (check_array)
        # 3. Oblicz macierz odległości
        # 4. Znajdź k najbliższych sąsiadów
        # 5. Głosowanie większościowe
        # ...

### 3.6 Weryfikacja

#### 3.6.1 Predykcja na danych syntetycznych

Dopasuj `KNNClassifier(n_neighbors=5)` do `X_train`, `y_train` z sekcji 3.2 i oblicz predykcje na `X_test`. Wyświetl dokładność (*accuracy*) i macierz pomyłek.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix


#### 3.6.2 Porównanie z `KNeighborsClassifier`

Porównaj predykcje swojego klasyfikatora z `sklearn.neighbors.KNeighborsClassifier` na tych samych danych i z tą samą wartością `n_neighbors`. Czy wyniki są identyczne?


In [ ]:
from sklearn.neighbors import KNeighborsClassifier


#### 3.6.3 `cross_val_score`

Prawidłowo zaimplementowany estymator powinien bezproblemowo współpracować z `cross_val_score`. Uruchom kroswalidację na połączonym zbiorze danych:


In [ ]:
from sklearn.model_selection import cross_val_score

X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

# Kroswalidacja z KNNClassifier
# ...

Jeśli w tym momencie pojawi się błąd — przeczytaj komunikat. Najczęstsze przyczyny:

- Brak `return self` w `fit()`.
- `check_is_fitted` zgłasza wyjątek, bo atrybuty wyuczone nie mają podkreślnika na końcu.
- `get_params()` nie działa poprawnie, bo nazwy argumentów w `__init__` nie zgadzają się z nazwami atrybutów.


## 4. Zbiór `Iris` — pełny test

Do tej pory testowaliśmy estymatory na danych syntetycznych. Sprawdźmy je na klasycznym zbiorze `Iris` — małym, ale rzeczywistym zbiorze danych z trzema klasami irysów i czterema cechami (długość i szerokość płatka oraz działki kielicha).


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print(f"Kształt: {X_iris.shape}, klasy: {np.unique(y_iris)}")

### 4.1 `KNNClassifier` na `Iris`

1. Podziel dane na zbiór treningowy (80%) i testowy (20%) za pomocą `train_test_split`.
2. Dopasuj `KNNClassifier(n_neighbors=5)` do zbioru treningowego.
3. Oblicz dokładność na zbiorze testowym.
4. Porównaj z `KNeighborsClassifier(n_neighbors=5)`.


In [ ]:
from sklearn.model_selection import train_test_split


### 4.2 Wpływ hiperparametru `n_neighbors`

Zbadaj, jak dokładność kroswalidacji zmienia się dla $k \in \{1, 3, 5, 7, 9, 11, 15, 21\}$. Dla każdej wartości $k$ oblicz średnią dokładność z 5-krotnej kroswalidacji (`cross_val_score`) i narysuj wykres.


### 4.3 `score()` i `get_params()`

`ClassifierMixin` automatycznie dostarcza metodę `score(X, y)`, która zwraca dokładność (*accuracy*).

1. Wywołaj `knn.score(X_test, y_test)` na dopasowanym klasyfikatorze. Czy wynik zgadza się z `accuracy_score(y_test, knn.predict(X_test))`?
2. Wywołaj `knn.get_params()`. Co zwraca? Skąd `BaseEstimator` wie, jakie hiperparametry ma klasa?
3. Wywołaj `knn.set_params(n_neighbors=3)` i ponownie oblicz `knn.score(X_test, y_test)`. Dlaczego wynik może być zaskakujący?

*Wskazówka do punktu 3:* `set_params` zmienia hiperparametr, ale nie wywołuje ponownie `fit()`. Aby nowa wartość $k$ wpłynęła na wyniki, trzeba ponownie dopasować model. To ważna obserwacja — `GridSearchCV` robi to automatycznie.


## 5. Zadanie dodatkowe: `KNNRegressor`

Algorytm k-NN działa również w problemach regresji — zamiast głosowania większościowego, predykcją jest średnia wartości docelowych $k$ najbliższych sąsiadów.

1. Zaimplementuj klasę `KNNRegressor`, dziedziczącą po `BaseEstimator` i `RegressorMixin`.
2. Przetestuj na danych syntetycznych z `make_regression`.
3. Porównaj z `sklearn.neighbors.KNeighborsRegressor`.

*Wskazówka:* `RegressorMixin` automatycznie dostarcza `score()` zwracające $R^2$.


In [ ]:
from sklearn.base import RegressorMixin
from sklearn.datasets import make_regression
